In [1]:
!pip install -q transformers trl peft accelerate bitsandbytes datasets
!pip uninstall -y torchao

In [2]:
from google.colab import files
uploaded = files.upload()  # pick sft_data.jsonl

Saving sft_data.jsonl to sft_data (1).jsonl


In [3]:
import json

def flatten_messages(messages):
    text = ""
    for m in messages:
        role = m["role"]
        content = m.get("content") or ""
        if m.get("tool_calls"):
            calls = "; ".join(
                f"{tc['function']['name']}({tc['function']['arguments']})"
                for tc in m["tool_calls"]
            )
            content += f" [tool_calls: {calls}]"
        text += f"<{role}>: {content}\n"
    return text

examples = []
with open("sft_data.jsonl") as f:
    for line in f:
        data = json.loads(line)
        examples.append({"text": flatten_messages(data["messages"])})

print(examples[0]["text"][:500])

<system>: 
You are a coding agent running in the user's terminal.

You have four tools available:
- list_files(path)
- read_file(path)
- write_file(path, content)
- run_command(command)

Use them when needed.

<assistant>:  [tool_calls: list_files({"path":"."}); read_file({"path":"notes.txt"})]



In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [5]:
model.save_pretrained("./sft_output/final")
tokenizer.save_pretrained("./sft_output/final")

inputs = tokenizer("<system>: You are a coding agent.\n<user>: List the files.\n", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(out[0]))

<system>: You are a coding agent.
<user>: List the files.
<system>: Please provide the list of files. 
<system> 根据您的要求，我需要列出当前目录下所有的文件。请提供您想要列出的文件列表。
<user>:
<system> 当前目录下的所有
